In [0]:
# =============================================================================
# CELL 1 — build workspace.bharosacare.pincodes_curated
# Source: india_post_pincode_directory (165,627 rows; one row per post office)
# Output: one row per pincode with a representative lat/lon + district/state.
#
# Data-quality issues handled (from Genie analysis):
#   - latitude/longitude stored as STRING  -> try_cast to double (Cell handles)
#   - 12,009 coords are the literal "NA"   -> try_cast turns them to null, and
#                                             the India bounding-box filter drops
#                                             them so they don't skew the median
#   - trailing/leading whitespace in text  -> trim district & statename
# Representative point = median (percentile_approx 0.5) of all valid office
# coords for that pincode — robust to a few stray/mis-typed coordinates.
# =============================================================================
from pyspark.sql import functions as F

PIN = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.india_post_pincode_directory"

pin = (spark.table(PIN)
       .withColumn("pincode",   F.col("pincode").cast("string"))
       .withColumn("district",  F.trim(F.col("district")))     # strip whitespace
       .withColumn("statename", F.trim(F.col("statename")))    # strip whitespace
       .withColumn("lat", F.expr("try_cast(latitude AS double)"))   # "NA"/garbage -> null
       .withColumn("lon", F.expr("try_cast(longitude AS double)"))) # "NA"/garbage -> null

# Keep only coords physically inside India; drops nulls and obvious bad points.
in_india = F.col("lat").between(6, 38) & F.col("lon").between(68, 98)

# Representative point per pincode = median of its valid office coordinates.
rep = (pin.filter(in_india).groupBy("pincode")
       .agg(F.expr("percentile_approx(lat, 0.5)").alias("rep_lat"),
            F.expr("percentile_approx(lon, 0.5)").alias("rep_lon")))

# District/state/office-count computed over ALL rows (not just ones with coords),
# so a pincode with only "NA" coords still gets its name metadata.
meta = (pin.groupBy("pincode")
        .agg(F.first("district",  ignorenulls=True).alias("district"),
             F.first("statename", ignorenulls=True).alias("state_name"),
             F.count("*").alias("n_offices")))

(meta.join(rep, "pincode", "left")   # left join: keep pincodes even if no valid coord
     .write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable("workspace.bharosacare.pincodes_curated"))

print("pincodes_curated rows:", spark.table("workspace.bharosacare.pincodes_curated").count())

In [0]:
# =============================================================================
# CELL 2 — build workspace.bharosacare.facilities_curated
# Source: facilities (10,088 rows). Output schema matches Jamie's db.py contract
# exactly (22 columns), so the Lakebase sync mirrors straight into app.facilities.
#
# Data-quality issues handled (from Genie analysis):
#   - 11 duplicate unique_id        -> de-dupe to one row per facility_id,
#                                       keeping the row with the most evidence
#   - 54 missing names              -> drop (can't display a nameless facility)
#   - yearEstablished 52% corrupt   -> keep only plausible 4-digit years, else null
#     ("null"/"true"/JSON blobs/single digits all become null)
#   - junk sentinels everywhere     -> "null"/"none"/"na"/"true"/"" etc. -> real null
#                                       (so they stop inflating completeness_score)
#   - numberDoctors/capacity garbage-> scrub blobs/non-numerics to null
#
# CONTRACT DECISION (overrides Genie's "convert to numeric" advice):
#   number_doctors / capacity / year_established stay STRING. Jamie's db.py
#   treats them as sparse display-only fields ("never used in a decision");
#   changing the type would break the A<->B column contract. We clean the
#   values but keep the type.
#
# CONTRACT FORMATTING (from Jamie's spec):
#   specialties / source_urls       -> normalized to ';'-delimited
#   pmjay_match                     -> always present, null until the stretch
# =============================================================================
from pyspark.sql import functions as F, Window
from functools import reduce

SRC  = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities"
JUNK = ["", "null", "none", "nan", "na", "n/a", "-", "true", "false"]   # sentinels -> null

# Trim + map any junk sentinel value to a real null.
def clean_text(name):
    t = F.trim(F.col(name).cast("string"))
    return F.when(F.lower(t).isin(*JUNK), F.lit(None).cast("string")).otherwise(t)

# Keep only a plausible 4-digit year (1800-2099); everything else -> null.
def clean_year(name):
    t = clean_text(name)
    return F.when(t.rlike(r"^(18|19|20)\d{2}$"), t).otherwise(F.lit(None).cast("string"))

# Keep short tokens that contain a digit; null out non-numerics and JSON blobs.
# (Stays STRING per the contract — we only remove garbage, not the type.)
def clean_count(name):
    t = clean_text(name)
    return F.when(t.rlike(r"\d") & (F.length(t) <= 40), t).otherwise(F.lit(None).cast("string"))

# Normalize a list-ish string to ';'-delimited: strip JSON brackets/quotes,
# collapse commas/pipes/semicolons/newlines to a single ';', trim stray ';',
# and turn an empty result into null (so it doesn't count as populated).
def to_semis(col):
    x = col.cast("string")
    x = F.regexp_replace(x, r'[\[\]"]', '')
    x = F.regexp_replace(x, r'\s*[,|;\n]+\s*', ';')
    x = F.regexp_replace(x, r'^;+|;+$', '')
    return F.when((x == "") | x.isNull(), F.lit(None).cast("string")).otherwise(x)

# --- select + rename raw columns into the clean contract names, cleaning inline ---
fac = spark.table(SRC).select(
    F.col("unique_id").cast("string").alias("facility_id"),
    clean_text("name").alias("name"),
    clean_text("address_stateOrRegion").alias("state"),
    clean_text("address_city").alias("city"),
    F.col("latitude").cast("double").alias("latitude"),    # numeric double (not string)
    F.col("longitude").cast("double").alias("longitude"),  # numeric double (not string)
    # pull the first 6-digit run as the pincode (handles "Pin: 110001" etc.)
    F.regexp_extract(F.trim(F.col("address_zipOrPostcode").cast("string")), r"(\d{6})", 1).alias("postcode"),
    to_semis(clean_text("specialties")).alias("specialties"),       # ';'-delimited
    clean_text("description").alias("description"),
    clean_text("capability").alias("capability"),
    clean_text("procedure").alias("procedure"),
    clean_text("equipment").alias("equipment"),
    clean_count("numberDoctors").alias("number_doctors"),           # cleaned, kept string
    clean_count("capacity").alias("capacity"),                      # cleaned, kept string
    clean_year("yearEstablished").alias("year_established"),        # valid year or null
    to_semis(clean_text("source_urls")).alias("source_urls"),       # ';'-delimited citations
)

# Drop nameless rows — a finder can't surface a facility with no name.
fac = fac.filter(F.col("name").isNotNull())

# De-duplicate facility_id: rank duplicates by how many evidence fields are
# populated and keep the richest (ties broken by name, then description) so the
# survivor is the most useful row, not an arbitrary one.
evid = ["description","capability","procedure","equipment","specialties","source_urls",
        "number_doctors","capacity","year_established"]
richness = reduce(lambda a, b: a + b, [F.col(c).isNotNull().cast("int") for c in evid])
w = Window.partitionBy("facility_id").orderBy(richness.desc(), F.col("name").asc(),
                                              F.col("description").asc_nulls_last())
fac = fac.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

# Join the pincode's district/state/representative point onto each facility.
pc = spark.table("workspace.bharosacare.pincodes_curated").select(
        "pincode",
        F.col("district").alias("pincode_district"),
        F.col("state_name").alias("pincode_state"),
        "rep_lat", "rep_lon")
j = fac.join(pc, fac.postcode == pc.pincode, "left").drop("pincode")

# Coordinate repair: if the facility's own coords are inside India use them
# (coord_source='facility'); else fall back to the pincode centroid
# (coord_source='pincode_fallback'); else leave unknown.
valid = F.col("latitude").between(6, 38) & F.col("longitude").between(68, 98)
j = (j.withColumn("coord_source",
        F.when(valid, F.lit("facility"))
         .when(F.col("rep_lat").isNotNull(), F.lit("pincode_fallback"))
         .otherwise(F.lit("unknown")))
      .withColumn("latitude",  F.when(valid, F.col("latitude")).otherwise(F.col("rep_lat")))
      .withColumn("longitude", F.when(valid, F.col("longitude")).otherwise(F.col("rep_lon"))))

# Normalize a state string for comparison (lowercase, '&'->'and', strip non-alnum)
# so "Tamil Nadu" == "tamilnadu" == "TAMIL-NADU".
def norm(c):
    x = F.lower(F.trim(c)); x = F.regexp_replace(x, r"&", "and")
    return F.regexp_replace(x, r"[^a-z0-9]", "")

# geo_status flags how trustworthy the location is:
#   repaired   = we used the pincode centroid (facility coords were bad)
#   unknown    = can't compare (missing a state on either side)
#   consistent = facility state matches the pincode's state
#   mismatch   = facility state disagrees with the pincode's state
j = j.withColumn("geo_status",
        F.when(F.col("coord_source") == "pincode_fallback", F.lit("repaired"))
         .when(F.col("pincode_state").isNull() | F.col("state").isNull(), F.lit("unknown"))
         .when(norm(F.col("state")) == norm(F.col("pincode_state")), F.lit("consistent"))
         .otherwise(F.lit("mismatch")))

# completeness_score (0..1): weighted coverage of evidence fields. Weights favor
# citations and clinical detail; sparse count fields contribute little. Because
# junk was nulled above, only genuinely populated fields score.
def has(c, wt):
    return F.when(F.col(c).isNotNull() & (F.length(F.trim(F.col(c).cast("string"))) > 0),
                  F.lit(wt)).otherwise(F.lit(0.0))
j = j.withColumn("completeness_score",
        has("source_urls",0.30)+has("specialties",0.20)+has("capability",0.15)+has("procedure",0.15)
       +has("equipment",0.10)+has("year_established",0.04)+has("number_doctors",0.03)+has("capacity",0.03))

# Final projection — exact column order/names for Jamie's contract.
# pmjay_match is null in the MVP (populated only if the stretch is built), but the
# column always exists so the synced schema is stable.
(j.select("facility_id","name","state","city","latitude","longitude","coord_source","postcode",
          "specialties","description","capability","procedure","equipment",
          "number_doctors","capacity","year_established","source_urls",
          "pincode_district","pincode_state","completeness_score","geo_status",
          F.lit(None).cast("string").alias("pmjay_match"))
  .write.mode("overwrite").option("overwriteSchema","true")
  .saveAsTable("workspace.bharosacare.facilities_curated"))

print("facilities_curated rows:", spark.table("workspace.bharosacare.facilities_curated").count())

In [0]:
# =============================================================================
# CELL 3 — sanity checks: confirm every cleaning strategy actually took effect.
# Each print states the expected value so a failure is obvious at a glance.
# =============================================================================
f = spark.table("workspace.bharosacare.facilities_curated")
p = spark.table("workspace.bharosacare.pincodes_curated")

n = f.count(); d = f.select("facility_id").distinct().count()
print(f"facilities_curated: {n} rows | {d} distinct ids | dup ids: {n-d} (expect 0)")  # de-dupe worked
print("pincodes_curated:", p.count(), "unique pincodes")
print("null names:", f.filter("name is null").count(), "(expect 0)")                    # nameless dropped
print("coords outside India:",
      f.filter("not (latitude between 6 and 38 and longitude between 68 and 98)").count(),
      "(expect 0)")                                                                      # coord repair worked
print("invalid years:",
      f.filter("year_established is not null and year_established not rlike '^(18|19|20)[0-9]{2}$'").count(),
      "(expect 0)")                                                                      # year scrub worked

display(f.groupBy("geo_status").count())                                                 # distribution sanity
display(f.selectExpr("round(min(completeness_score),2) lo","round(max(completeness_score),2) hi"))
# eyeball the cleaned text + ';'-delimited fields:
display(f.select("name","specialties","source_urls","number_doctors","year_established",
                 "completeness_score","geo_status").limit(8))

In [0]:
# =============================================================================
# CELL 4 — contract check against Jamie's db.py spec.
# PASS = "missing" and "extra" both empty, and latitude/longitude show 'double'.
# =============================================================================
expected = ["facility_id","name","state","city","latitude","longitude","coord_source","postcode",
            "specialties","description","capability","procedure","equipment",
            "number_doctors","capacity","year_established","source_urls",
            "pincode_district","pincode_state","completeness_score","geo_status","pmjay_match"]
f = spark.table("workspace.bharosacare.facilities_curated")
actual = [fld.name for fld in f.schema.fields]
print("missing vs spec:", [c for c in expected if c not in actual])
print("extra vs spec  :", [c for c in actual if c not in expected])
for fld in f.schema.fields:
    print(f"  {fld.name}: {fld.dataType.simpleString()}")